In [1]:
import numpy as np
import pandas as pd
import torch
import os
import time

from transformers import Trainer, TrainingArguments
from datasets import load_dataset
from peft import PromptTuningConfig, get_peft_model
from peft import LoraConfig, get_peft_model, TaskType

from metrics import calculate_meteor_score, calculate_bleu_score
from tools import (inference_batch, get_model, tokenize_function, print_number_of_trainable_model_parameters,
                   format_inference_shots, format_input_texts)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/home/denis/ax/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt_tab to /home/denis/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/denis/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Tarefa 1

## <span style="color:#FFA500"> Utilizando novamente o dataset de TESTE e o modelo ajustado na tarefa 2, execute mais uma vez as instruções da coluna “instruction” do dataset. Extraia 2 partes do dataset com 3000 linhas cada uma. Uma das partes será o dataset de treino e a outra, o de teste.Você pode escolher as linhas de cada dataset da forma que preferir.</span>

In [2]:
dataset_name = 'tatsu-lab/alpaca'

dataset = load_dataset(dataset_name)
dataset['train'].num_rows

52002

In [3]:
test_size = 3000
train_size = 3000
inference_input_idx = 5

idx = np.random.choice(dataset['train'].shape[0], size=train_size + test_size + inference_input_idx, replace=False)
traind_idx = idx[:train_size]
test_idx = idx[train_size:-inference_input_idx]
inference_input_idx = idx[-inference_input_idx:]
train_dataset = dataset.copy()['train'].select(traind_idx)
test_dataset = dataset.copy()['train'].select(test_idx)
inference_dataset = dataset.copy()['train'].select(inference_input_idx)

print(f'Train data size: {len(train_dataset)}\nTest data size: {len(test_dataset)}\nInference data size: {len(inference_dataset)}')

Train data size: 3000
Test data size: 3000
Inference data size: 5


## <span style="color:#FFA500">Utilizando o modelo indicado, execute as instruções da coluna “instruction” do dataset de TESTE.</span>

### Sem usar template de prompt

In [4]:
model, tokenizer = get_model(device)
print(print_number_of_trainable_model_parameters(model))

trainable model parameters: 124439808
all model parameters: 124439808
percentage of trainable model parameters: 100.00%


In [5]:
result_df = pd.DataFrame()
result_df['instruction'] = test_dataset['instruction']
result_df['reference'] = test_dataset['output']
result_df['prediction'] = inference_batch(test_dataset['instruction'], model, tokenizer, device)

Progress: [#############################] [3000/3000]

In [6]:
for i in result_df.index[:10]:
    print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
    print('\n' + ('-' * 150))

Instruction: Edit the sentence, “A king can no longer test the mettle of his knights by challenge to single combat” to make it easier to read.
Response: ‧ That, and the fact that you only need to read one sentence in order to take part in this event, to take advantage of our rewards.›

"Alright, so it looks like the rest of your team did it, as well?"

At the top of the page, the screen showed three "BATTLE" words in short sentences, each containing a different combatant, one of which was named "Archie".

It was a short message.

The only difference in the sentence was that the "Archer" text was in a different language, while the other two words were in English.



If you click on it, you will be taken to the battle page in English, but you won't be able to get a better idea of what was happening as you've entered the battle.


For example, if you click the picture, the text of the battle will be blacked out (without "Archery", "
Reference: A king can no longer assess the strength and 

In [7]:
general_results = {}
result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

general_results['without_prompt_and_ft'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                          axis=1).mean().to_dict()
general_results['without_prompt_and_ft']

{'bleu': 0.003931185484904012, 'meteor': 0.14030528065667675}

In [8]:
result_df.head()

,instruction,reference,prediction,bleu,meteor
0,"Edit the sentence, “A king can no longer test ...",A king can no longer assess the strength and c...,"‧ That, and the fact that you only need to rea...",0.001705,0.094851
1,What is the area of a triangle with base 8 cm ...,The area of the triangle is 20 cm squared.,A triangle is not a straight line. A triangle ...,0.002702,0.168278
2,Create a timeline to showcase important events...,1943: The first paper on AI is published by Wa...,"The first event will be hosted on May 1st, whe...",0.008404,0.138151
3,Compare and contrast militarism and imperialism,Militarism and imperialism are both forms of a...,"to a more political view of things. The ""polit...",0.002259,0.189445
4,Create a story about a student who had an upco...,Jack was a hardworking student who was determi...,The class consisted of a computer game called ...,0.008605,0.201317


## <span style="color:#FFA500">Avalie a qualidade do resultado. As respostas estavam corretas? Quais métodos podem ser usados para melhorá-las?</span>

---
<span style="color:#00BFFF">Resposta: Tanto os valores retornados nas métricas de avaliação, quanto os outputs gerados pelo modelo, não são bons... Isso já era um comportamento esperado, pois o modelo "ComCom/gpt2-small" além de pequeno foi treinado apenas para completar texto. Podemos utilizar métodos de prompt engineering, inserir shots de inferência dentro dos inputs para aumentar a chance de o modelo entender o padrão das tarefas solicitadas.</span>

### Usando template de prompt

#### Zero-shot inference

In [9]:
n_shots = 0
inference_shots = format_inference_shots(inference_dataset, n_shots)
print(inference_shots)

In [10]:
input_texts_0s = format_input_texts(test_dataset, inference_shots)

result_df = pd.DataFrame()
result_df['instruction'] = test_dataset['instruction']
result_df['reference'] = test_dataset['output']
result_df['prediction'] = inference_batch(input_texts_0s, model, tokenizer, device)

Progress: [#############################] [3000/3000]

In [11]:
for i in result_df.index[:10]:
    print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
    print('\n' + ('-' * 150))

Instruction: Edit the sentence, “A king can no longer test the mettle of his knights by challenge to single combat” to make it easier to read.
Response: Write an input.
...


In a similar manner to the command below, you will need to send the following messages, and they will be provided by the above command.
The first message would be to read a response from the command at the prompt. When you type "send", it's to send a message that your browser has set up.
This is a very simple command. As mentioned in the previous example, a "command" is a file in your browser and must contain the following information:
The target command (to read a reply) is to read it from the browser and to send it to the server. This is done by using a file extension called "reply.json". To send the file you would use this command. By adding the following to the output of "send":


{ "name": "John", "type": "file", "response": { "type" : "file" } }

The second message would simply
Reference: A king can no longe

In [12]:
result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

general_results['prompt_eng_0shot'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                          axis=1).mean().to_dict()
general_results['prompt_eng_0shot']

{'bleu': 0.003676067222122717, 'meteor': 0.12885171299150597}

In [13]:
result_df.head()

,instruction,reference,prediction,bleu,meteor
0,"Edit the sentence, “A king can no longer test ...",A king can no longer assess the strength and c...,Write an input.\n...\n\n\nIn a similar manner ...,0.001779,0.104167
1,What is the area of a triangle with base 8 cm ...,The area of the triangle is 20 cm squared.,What has been written for this task? The first...,0.002099,0.079681
2,Create a timeline to showcase important events...,1943: The first paper on AI is published by Wa...,Create an action on the timeline that will sho...,0.002435,0.152749
3,Compare and contrast militarism and imperialism,Militarism and imperialism are both forms of a...,"## Response is good, and it is done with a sim...",0.004310,0.158831
4,Create a story about a student who had an upco...,Jack was a hardworking student who was determi...,A student with a previous assignment at an ESL...,0.005599,0.236481


#### One shot inference

In [14]:
n_shots = 1
inference_shots = format_inference_shots(inference_dataset, n_shots)
print(inference_shots)

Below are 1 instructions that describe task resolutions. First comes ### Instruction, providing the instruction to be followed. Second comes ### Input, providing information that supports the instruction. And last comes ### Response, bringing the response to this instruction.
### Instruction:
Develop a research question that can be answered using data.

### Input:
Traffic congestion

### Response:
What factors contribute to traffic congestion in city centers?




In [15]:
input_texts_1s = format_input_texts(test_dataset, inference_shots)

result_df = pd.DataFrame()
result_df['instruction'] = test_dataset['instruction']
result_df['reference'] = test_dataset['output']
result_df['prediction'] = inference_batch(input_texts_1s, model, tokenizer, device)

Progress: [#############################] [3000/3000]

In [16]:
for i in result_df.index[:10]:
    print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
    print('\n' + ('-' * 150))

Instruction: Edit the sentence, “A king can no longer test the mettle of his knights by challenge to single combat” to make it easier to read.
Response: The queen has a high desire to see one's children and can no more take it away than she can, so she will need some money to pay for her children's education. The king can then send her a letter by mail asking if the child is suitable for education, or if his parents are willing to pay.
"If my child is a good child who is at home and will meet you in the summer, I will send him to a well-off and well-educated woman, so that you may not be disappointed in his education. I will also give you a suitable place to live."

###### Response: The answer is "Yes, but you are not allowed to marry in a well placed place like the palace or your home." ### Response:

.
.



# Response: (to a servant)

"Sir, how about this? I can't marry."


.


"Why should I?"


The answer
Reference: A king can no longer assess the strength and courage of his knights

In [17]:
result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

general_results['prompt_eng_1shot'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                          axis=1).mean().to_dict()
general_results['prompt_eng_1shot']

{'bleu': 0.0037120767131626972, 'meteor': 0.12941832805613349}

In [18]:
result_df.head()

,instruction,reference,prediction,bleu,meteor
0,"Edit the sentence, “A king can no longer test ...",A king can no longer assess the strength and c...,The queen has a high desire to see one's child...,0.004202,0.168122
1,What is the area of a triangle with base 8 cm ...,The area of the triangle is 20 cm squared.,What are the height of the triangle with the b...,0.008016,0.142857
2,Create a timeline to showcase important events...,1943: The first paper on AI is published by Wa...,How important is this event to the system?\n\n...,0.004401,0.131323
3,Compare and contrast militarism and imperialism,Militarism and imperialism are both forms of a...,What is the best way to organize war?\n\n## Re...,0.004166,0.178082
4,Create a story about a student who had an upco...,Jack was a hardworking student who was determi...,"If the answer doesn't follow the goal, how wou...",0.003369,0.097336


### Few-shot inference

In [ ]:
n_shots = 3
inference_shots = format_inference_shots(inference_dataset, n_shots)
print(inference_shots)

Below are 5 instructions that describe task resolutions. First comes ### Instruction, providing the instruction to be followed. Second comes ### Input, providing information that supports the instruction. And last comes ### Response, bringing the response to this instruction.
### Instruction:
Develop a research question that can be answered using data.

### Input:
Traffic congestion

### Response:
What factors contribute to traffic congestion in city centers?


### Instruction:
Select three websites to research a topic.

### Input:


### Response:
Wikipedia, BBC, and The New York Times are three good sources for researching any topic.


### Instruction:
Give three example of plants that thrive in shade.

### Input:


### Response:
Examples of plants that thrive in shade include hostas, ferns, and impatiens. Hostas have large, colorful leaves and come in a variety of shapes and sizes. Ferns are often sold in pots and love shady areas. Impatiens are known for their bright colors, come in

In [21]:
input_texts_fs = format_input_texts(test_dataset, inference_shots)

result_df = pd.DataFrame()
result_df['instruction'] = test_dataset['instruction']
result_df['reference'] = test_dataset['output']
result_df['prediction'] = inference_batch(input_texts_fs, model, tokenizer, device, batch_size=30)

Progress: [###################################################                                                ] [1560/3000]

KeyboardInterrupt: 

In [ ]:
for i in result_df.index[:10]:
    print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
    print('\n' + ('-' * 150))

In [ ]:
result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

general_results['prompt_eng_fs'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                          axis=1).mean().to_dict()
general_results['prompt_eng_fs']

In [ ]:
result_df.head()

## <span style="color:#FFA500">Avalie a qualidade do resultado. As respostas estavam corretas? Quais métodos podem ser usados para melhorá-las?</span>


---
<span style="color:#00BFFF">Resposta: Ao usar um template de prompt, os resultados...</span>

# Tarefa 2

## <span style="color:#FFA500"> Utilizando o dataset de TREINO, faça o ajuste (fine-tuning) do modelo. Você pode usar técnicas para simplificar o fine-tuning, ajustando um conjunto menor de parâmetros e consumindo menos memória.</span>

## <span style="color:#FFA500"> Utilizando novamente o dataset de TESTE e o modelo ajustado na tarefa 2, execute mais uma vez as instruções da coluna “instruction” do dataset.</span>

# Fine Tuning

In [ ]:
model, tokenizer = get_model(device)

In [ ]:
prompt = """Below is an instruction for a task. First comes ### Instruction, giving the instruction to be followed. Second comes ### Input, giving information that supports the instruction. Write a response that adequately completes the request from ### Response.
### Instruction:
{}

### Input:
{}

### Response:
"""

tokenized_train_dataset = train_dataset.map(lambda example: tokenize_function(example, prompt, tokenizer), batched=False)

### Prompt Tuning with Prompt template

In [ ]:
num_virtual_tokens = 30

peft_config = PromptTuningConfig(
    task_type="CAUSAL_LM",
    num_virtual_tokens=num_virtual_tokens,  # Número de tokens do prompt
    token_dim=model.config.hidden_size,         # Dimensão do embedding (768 para GPT-2 small)
    num_attention_heads=model.config.num_attention_heads # Número de heads de atenção do modelo
)

prompt_tuning_model = get_peft_model(model, peft_config)

In [ ]:
print(print_number_of_trainable_model_parameters(prompt_tuning_model))

In [ ]:
output_dir = f'./prompt-tuning-gpt2-alpaca-{str(int(time.time()))}'

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    learning_rate=1e-3,
    logging_steps=100,
    max_steps=100,  # Aumentado para um valor mais útil
    remove_unused_columns=False,  # Importante para PEFT
    label_names=["input_ids"]  # Resolve o warning
)
    
peft_trainer = Trainer(
    model=prompt_tuning_model,
    args=peft_training_args,
    train_dataset=tokenized_train_dataset,
)

In [ ]:
peft_trainer.train()
peft_trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir, safe_serialization=True)

In [ ]:
input_types = {'0s': input_texts_0s,
               '1s': input_texts_1s,
               'fs': input_texts_fs
               }

for key, input_texts in input_types.items():
    result_df = pd.DataFrame()
    result_df['instruction'] = test_dataset['instruction']
    result_df['reference'] = test_dataset['output']
    result_df['prediction'] = inference_batch(input_texts, prompt_tuning_model, tokenizer, device, batch_size=30)

    result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
    result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

    general_results[f'prompt_tunning_{key}'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                            axis=1).mean().to_dict()
    general_results[f'prompt_tunning_{key}']

    for i in result_df.index[:3]:
        print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
        print('\n' + ('-' * 150))
    print('\n' + ('=' * 150))

In [ ]:
result_df.head()

### LoRA fine tuning with Prompt template

In [ ]:
model, tokenizer = get_model(device)

In [ ]:
lora_config = LoraConfig(
    r=512,
    lora_alpha=256,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.1,
    bias="lora_only",
    task_type=TaskType.CAUSAL_LM
)

In [ ]:
peft_model = get_peft_model(model, lora_config)

In [ ]:
print(print_number_of_trainable_model_parameters(peft_model))

In [ ]:
output_dir = f'./loraft-gpt2-alpaca-{str(int(time.time()))}'

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    save_safetensors=False,
    learning_rate=1e-3, # Taxa deve ser maior que no full fine tunning
    logging_steps=100,
    max_steps=100   
)
    
peft_trainer = Trainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=tokenized_train_dataset,
)


In [ ]:
peft_trainer.train()
peft_trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir, safe_serialization=True)

In [ ]:
for key, input_texts in input_types.items():
    result_df = pd.DataFrame()
    result_df['instruction'] = test_dataset['instruction']
    result_df['reference'] = test_dataset['output']
    result_df['prediction'] = inference_batch(input_texts, peft_model, tokenizer, device, batch_size=30)

    result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
    result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

    general_results[f'prompt_eng_and_lora_{key}'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                            axis=1).mean().to_dict()
    general_results[f'prompt_eng_and_lora_{key}']

    for i in result_df.index[:3]:
        print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
        print('\n' + ('-' * 150))
    print('\n' + ('=' * 150))

### Full fine tuning with Prompt template

In [ ]:
model, tokenizer = get_model(device)

In [ ]:
print(print_number_of_trainable_model_parameters(model))

In [ ]:
output_dir = f'./fullft-gpt2-alpaca-{str(int(time.time()))}'

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    save_safetensors=False,
    learning_rate=1e-4,
    logging_steps=100,
    max_steps=100
)
    
trainer = Trainer(
    model=model,
    args=peft_training_args,
    train_dataset=tokenized_train_dataset,
)


In [ ]:
trainer.train()
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir, safe_serialization=True)

In [ ]:
for key, input_texts in input_types.items():
    result_df = pd.DataFrame()
    result_df['instruction'] = test_dataset['instruction']
    result_df['reference'] = test_dataset['output']
    result_df['prediction'] = inference_batch(input_texts, model, tokenizer, device)

    result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
    result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

    general_results[f'prompt_eng_and_full_ft_{key}'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                            axis=1).mean().to_dict()
    general_results[f'prompt_eng_and_full_ft_{key}']

    for i in result_df.index[:3]:
        print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
        print('\n' + ('-' * 150))
    print('\n' + ('=' * 150))

## <span style="color:#FFA500">Interprete os resultados em comparação com os obtidos na tarefa 1.</span>

In [ ]:
general_results_df = pd.DataFrame(general_results).T
general_results_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 10))
for i, metric in enumerate(general_results_df.columns):
    plt.subplot(1,2,i+1)
    sns.barplot(general_results_df, x=metric)
    # general_results_df.T[metric].plot(kind='bar')

<span style="color:#00BFFF">Resposta: Mesmo após os processos de fine tuning testados, os resultados não melhoraram o suficiente para que o modelo se adaptasse a seguir as instruções. Além disso, o conjunto de dados "tatsu-lab/alpaca" traz diversos tipos de tarefas diferentes (não categorizadas), dificultando ainda mais o fine tuning do modelo para tarefas específicas. Seria necessário termos muitos registros para cada tipo específico de tarefa, para que o modelo ficasse bom em todas elas. Uma alternativa de melhoria seria aumentar o conjunto de dados utilizados para treinamento, outra seria utilizar few-shot inferences com inferências específicas para a tarefa que deve ser solucionada.</span>

## <span style="color:#FFA500">Se você precisasse agrupar as perguntas que tratam de assuntos semelhantes nos 2 datasets, como faria?</span>
---
<span style="color:#00BFFF">Resposta: Poderia gerar embeddings semânticos das entradas da coluna "instruction" usando um modelo de embeddings semântico, que transforma cada frase em um vetor numérico representando seu significado. Com esses vetores, é possível aplicar um algoritmo de clusterização (como KMeans, DBSCAN ou HDBSCAN) para identificar grupos de instruções com temas ou propósitos parecidos, como traduções, resumos ou geração de código. Isso permite organizar o dataset por tipo de tarefa, facilitando análises, seleção de exemplos e melhorias no modelo.</span>